<a href="https://colab.research.google.com/github/e23323-dot/Statistical-Learning-e23323/blob/main/Assignment_07d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import beta, norm
import plotly.io as pio
pio.renderers.default = 'colab'

print("Q3. Bayesian Estimations for Structural Health Monitoring")
print("="*70)

print("""
1. Prior Belief Boundaries:

Θ ~ Beta(8, 1.5) on θ ∈ (0, 1]

Expected Prior Stiffness:
E[Θ] = α / (α + β) = 8 / (8 + 1.5) = 8 / 9.5 = 0.8421

Why this prior is appropriate:
- Mean ≈ 0.84 indicates high initial belief of structural health
- Small variance (from α+β = 9.5) shows confidence in manufacturing quality
- Bounded support [0, 1] matches physical constraint of stiffness efficiency
- β < α creates left-skewed distribution (slightly more mass near 1.0)
""")

theta_grid = np.linspace(0.01, 1.0, 500)
prior = beta.pdf(theta_grid, 8, 1.5)
prior /= np.trapezoid(prior, theta_grid)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior, mode='lines', name='Beta(8,1.5) Prior'))
fig.add_vline(x=8/9.5, line_dash="dash", line_color="red", annotation_text="E[θ] = 0.842")
fig.update_layout(title='SHM Prior Distribution', xaxis_title='θ (Stiffness Efficiency)', yaxis_title='Density', height=400, width=800)
fig.show()

print(f"E[θ] = {8/(8+1.5):.4f}")

print("""
2. Structural Likelihood Formulation:

Measurement Model:
y_k = θ · K_nominal · e^{ε_k}, where ε_k ~ N(0, σ²)

Taking natural logarithm:
ln(y_k) = ln(θ) + ln(K_nominal) + ε_k

Therefore, conditional on θ:
ln(y_k) | θ ~ N(ln(θ) + ln(K_nominal), σ²)

Likelihood function for a single measurement:
L(y_k | θ) = 1 / (y_k · σ · √(2π)) · exp[- (ln(y_k / (θ · K_nominal)))² / (2σ²)]

Joint Likelihood for running history:
L(y^(k) | θ) = ∏_{i=1}^k L(y_i | θ)

3. Mathematical Formulation of Non-Conjugate Grid Update:

Why no closed-form solution exists:
- Prior is Beta: f(θ) ∝ θ^{α-1} (1-θ)^{β-1}
- Likelihood is Log-normal: L(y|θ) ∝ (1/θ) · exp[- (ln(y/(θ·K_nominal)))²/(2σ²)]
- Product: Beta × Log-normal does not simplify to any standard distribution family
- No conjugate prior exists for this likelihood

Recursive posterior update (up to proportionality):
f(θ | y^(k)) ∝ L(y_k | θ) · f(θ | y^(k-1))

4. Running Point Estimates:

Posterior Mean (Bayes estimate):
θ̂_Bayes^(k) = E[Θ | y^(k)] = ∫₀¹ θ · f(θ | y^(k)) dθ

Maximum A Posteriori (MAP) estimate:
θ̂_MAP^(k) = argmax_{θ ∈ (0,1]} f(θ | y^(k))

5. Algorithmic Grid Approximation and Normalization:

Step-by-step numerical procedure:

1. Define a discrete grid over θ ∈ (0, 1]:
   Θ_grid = {θ_1, θ_2, ..., θ_M} with uniform spacing Δθ

2. Initialize prior density on the grid:
   p_0(θ_j) = Beta(θ_j | 8, 1.5)

3. Normalize the prior using trapezoidal rule:
   p_0(θ_j) = p_0(θ_j) / [Σ_{m=1}^M p_0(θ_m) · Δθ]

4. For each new sensor reading y_k (k = 1, 2, ..., n):

   a. Compute likelihood at each grid point:
      L(y_k | θ_j) = 1/(y_k·σ·√(2π)) · exp[- (ln(y_k/(θ_j·K_nominal)))²/(2σ²)]
      (Set L = 0 where expression is invalid)

   b. Compute unnormalized posterior:
      g(θ_j) = L(y_k | θ_j) · p_{k-1}(θ_j)

   c. Normalize using trapezoidal rule:
      Z_k = Σ_{j=1}^M g(θ_j) · Δθ
      p_k(θ_j) = g(θ_j) / Z_k

   d. Handle boundaries:
      - θ = 0 is excluded (log undefined); use small epsilon (e.g., 0.001)
      - θ = 1 is included as the healthy state

   e. Compute point estimates:
      θ̂_Bayes^(k) = Σ_{j=1}^M θ_j · p_k(θ_j) · Δθ
      θ̂_MAP^(k) = θ_j* where p_k(θ_j*) is maximal

6. Performance Tracking and Degradation Convergence Analysis:
""")

def simulate_shm(n=15, theta_true=0.68, K_nominal=50.0, sigma=0.15):
    np.random.seed(42)

    theta_grid = np.linspace(0.01, 1.0, 300)
    delta_theta = theta_grid[1] - theta_grid[0]
    prior = beta.pdf(theta_grid, 8, 1.5)
    prior /= np.trapezoid(prior, theta_grid)

    mean_est = []
    map_est = []
    posterior_curves = []

    posterior_curves.append(prior)

    for k in range(n):
        eps = np.random.normal(0, sigma)
        y_k = theta_true * K_nominal * np.exp(eps)

        log_ratio = np.log(y_k / (theta_grid * K_nominal))
        likelihood = (1/(y_k * sigma * np.sqrt(2*np.pi))) * np.exp(-log_ratio**2/(2*sigma**2))
        likelihood[np.isnan(likelihood)] = 0

        posterior = prior * likelihood
        posterior /= np.trapezoid(posterior, theta_grid)

        mean_est.append(np.trapezoid(theta_grid * posterior, theta_grid))
        map_est.append(theta_grid[np.argmax(posterior)])

        if k in [0, 1, 2, 5, 10, 14]:
            posterior_curves.append(posterior)

        prior = posterior

    return mean_est, map_est, posterior_curves

mean_est, map_est, posterior_curves = simulate_shm()

fig1 = go.Figure()
milestones = [0, 1, 2, 5, 10, 15]
colors = ['blue', 'cyan', 'green', 'yellow', 'orange', 'red']
for i, (milestone, curve) in enumerate(zip(milestones, posterior_curves)):
    fig1.add_trace(go.Scatter(x=theta_grid, y=curve, mode='lines', name=f'k={milestone}', line=dict(color=colors[i])))

fig1.add_vline(x=0.68, line_dash="dash", line_color="black", annotation_text="θ_true = 0.68")
fig1.update_layout(title='SHM: Posterior Density Evolution', xaxis_title='θ (Stiffness Efficiency)', yaxis_title='Density', height=500, width=800)
fig1.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(range(16)), y=[0.68]*16, mode='lines', name='θ_true = 0.68', line=dict(dash='dash')))
fig2.add_trace(go.Scatter(x=list(range(1,16)), y=mean_est, mode='lines+markers', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=list(range(1,16)), y=map_est, mode='lines+markers', name='MAP'))
fig2.update_layout(title='SHM: Convergence of Estimators', xaxis_title='Inspection Step k', yaxis_title='θ Estimate', height=500, width=800)
fig2.show()

print("""
Analysis:
- The initial optimistic prior (E[θ] = 0.842) is overcome after approximately 5-7 readings
- The posterior distribution narrows significantly after approximately 10 readings
- MAP converges faster than Mean (MAP mode moves more quickly to θ_true)
- Narrowing density implies engineers can confidently set safety thresholds
- At k = 15, the distribution is tightly concentrated near θ = 0.68
- The system confidently isolates the 68% damage state after about 7-8 readings
- The narrowing of density curves indicates increased certainty, allowing precise safety threshold determination
""")

Q3. Bayesian Estimations for Structural Health Monitoring

1. Prior Belief Boundaries:

Θ ~ Beta(8, 1.5) on θ ∈ (0, 1]

Expected Prior Stiffness:
E[Θ] = α / (α + β) = 8 / (8 + 1.5) = 8 / 9.5 = 0.8421

Why this prior is appropriate:
- Mean ≈ 0.84 indicates high initial belief of structural health
- Small variance (from α+β = 9.5) shows confidence in manufacturing quality
- Bounded support [0, 1] matches physical constraint of stiffness efficiency
- β < α creates left-skewed distribution (slightly more mass near 1.0)



E[θ] = 0.8421

2. Structural Likelihood Formulation:

Measurement Model:
y_k = θ · K_nominal · e^{ε_k}, where ε_k ~ N(0, σ²)

Taking natural logarithm:
ln(y_k) = ln(θ) + ln(K_nominal) + ε_k

Therefore, conditional on θ:
ln(y_k) | θ ~ N(ln(θ) + ln(K_nominal), σ²)

Likelihood function for a single measurement:
L(y_k | θ) = 1 / (y_k · σ · √(2π)) · exp[- (ln(y_k / (θ · K_nominal)))² / (2σ²)]

Joint Likelihood for running history:
L(y^(k) | θ) = ∏_{i=1}^k L(y_i | θ)

3. Mathematical Formulation of Non-Conjugate Grid Update:

Why no closed-form solution exists:
- Prior is Beta: f(θ) ∝ θ^{α-1} (1-θ)^{β-1}
- Likelihood is Log-normal: L(y|θ) ∝ (1/θ) · exp[- (ln(y/(θ·K_nominal)))²/(2σ²)]
- Product: Beta × Log-normal does not simplify to any standard distribution family
- No conjugate prior exists for this likelihood

Recursive posterior update (up to proportionality):
f(θ | y^(k)) ∝ L(y_k | θ) · f(θ | y^(k-1))

4. Running Point Estimates:

Posterior Mean (Bayes estimate):
θ̂_Bayes^(k) = E[Θ | y^


Analysis:
- The initial optimistic prior (E[θ] = 0.842) is overcome after approximately 5-7 readings
- The posterior distribution narrows significantly after approximately 10 readings
- MAP converges faster than Mean (MAP mode moves more quickly to θ_true)
- Narrowing density implies engineers can confidently set safety thresholds
- At k = 15, the distribution is tightly concentrated near θ = 0.68
- The system confidently isolates the 68% damage state after about 7-8 readings
- The narrowing of density curves indicates increased certainty, allowing precise safety threshold determination

